# WebQSP: visual, storage-conscious GRAG preprocessing on Kaggle

Enable **Internet** and **GPU T4 x2** in Kaggle Settings. Run cells top to bottom.
This notebook uses Kaggle's own Python/PyTorch/CUDA; **no venv, uv, CUDA replacement,
Llama download, gated-model token, or torch-scatter build is needed**.
Push/upload the updated helper script with this notebook before cloning it on Kaggle.

Pipeline: raw triples -> text tables -> batched SentenceBERT embeddings (one query graph
in RAM per worker) -> directed K-hop ego-graph pooling -> cosine ranking -> top centers
-> CSV hard prompt + compact retrieved graph -> lazy expansion into GraphLLM inputs.
No full embedded-graph directory, repeated question matrices, or duplicate ego features
are saved. Two independent workers use one T4 each; GPU memory is **not pooled**.
Retrieval runs on CPU between embedding jobs, so both GPUs need not stay 100% busy.

This is preprocessing/visualization, not Llama training or a claim of paper reproduction.
Native package versions and optional FP16 can change floating-point results.


## Fidelity and storage choices

Preserve HF train/validation/test membership and original source indices, lowercase entity text,
and use `k=2`, `topk=20`, `topk_entity=10`, `augment="none"`.
Preserve the author's unusual removal of **all** edges for repeated endpoint pairs.
Unlike the legacy retrieval helper, text rows stay aligned after this filtering,
isolated nodes have an explicit node count, and single-center/edgeless outputs
receive valid query fields. There are no silent exception fallbacks.

`topk_entity` limits the CSV hard-prompt rows, **not** the soft graph size.
Compact storage does not cap nodes, edges, centers, or hops; a very large graph
can still exhaust RAM/time. Start with two samples and inspect the estimates.

Some source questions have `graph=[]`. This graph-only experiment explicitly excludes them and writes every exclusion to `excluded.json`; it does not invent graph nodes or silently discard data. Retained samples get contiguous processing `id` values, while `source_index`, native `source_id`, `source_split`, and `source_split_index` preserve their original identities. `prepared.json` records original and filtered split mappings/counts. Evaluation uses the retained subset, **not the original full-dataset denominator**, so report the exclusions with any results.


## 1. Clone once (or reuse the checkout)


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

PY = sys.executable
ROOT = Path("/kaggle/working/BioGRAG-QA")
REPOSITORY = "https://github.com/MichaelPham2005/BioGRAG-QA.git"
BRANCH = "GRAG/reproduce/WebQSP"  # Change if your updated files are on another branch.
if not ROOT.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPOSITORY, str(ROOT)], check=True)
if not (ROOT / ".git").is_dir():
    raise RuntimeError(f"{ROOT} exists but is not a git checkout; inspect it first.")
SCRIPT = ROOT / "src/utils/notebook_preprocessing/kaggle_preprocess.py"
if not SCRIPT.is_file():
    raise FileNotFoundError("Push/pull or upload the updated kaggle_preprocess.py first.")
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
print("Source commit:", COMMIT)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HOME", "/tmp/grag-hf-cache")


## 2. Keep Kaggle's environment; install missing preprocessing packages only

No package upgrades or author-version pins. Import failures are explicit; if an installed
package is incompatible, inspect the traceback rather than replacing torch/CUDA blindly.
`/tmp/grag-hf-cache` stores public HF downloads for this session, not a second raw Arrow
snapshot or a saved notebook output. Its files still consume available disk space.


In [ ]:
import importlib.util

if importlib.util.find_spec("torch") is None:
    raise RuntimeError("Kaggle's torch is missing. Select the GPU notebook image and restart.")
packages = {
    "torch_geometric": "torch_geometric", "transformers": "transformers",
    "datasets": "datasets", "pandas": "pandas", "sklearn": "scikit-learn",
    "matplotlib": "matplotlib", "networkx": "networkx",
}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([PY, "-m", "pip", "install", "-q", *missing], check=True)
    importlib.invalidate_caches()

import json
import torch
import pandas as pd
import transformers
import torch_geometric
from huggingface_hub import HfApi
from torch_geometric.nn import GATConv

print("Python:", sys.version, "torch:", torch.__version__, "CUDA:", torch.version.cuda)
print("transformers:", transformers.__version__, "PyG:", torch_geometric.__version__)
if not torch.cuda.is_available() or torch.cuda.device_count() != 2:
    raise RuntimeError("Select Kaggle GPU T4 x2 and restart the session.")
for gpu in range(2):
    with torch.cuda.device(gpu):
        print(gpu, torch.cuda.get_device_name(gpu))
        conv = GATConv(4, 4, edge_dim=4).to(f"cuda:{gpu}")
        result = conv(torch.ones(2, 4, device=f"cuda:{gpu}"),
                      torch.tensor([[0], [1]], device=f"cuda:{gpu}"),
                      torch.ones(1, 4, device=f"cuda:{gpu}"))
        assert result.shape == (2, 4) and torch.isfinite(result).all()
        del conv, result
        torch.cuda.empty_cache()

spec = importlib.util.spec_from_file_location("kaggle_preprocess", SCRIPT)
pipeline = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pipeline)


## 3. Prepare text tables, source IDs, and filtered split IDs (no embeddings yet)

A fresh directory is prepared once. Reuse it to resume; use a different directory
if raw inputs change. The manifest fingerprints the source and table contents.
Text preparation does not import `src.dataset` or load a model as a side effect.
The graph-only directory names below separate these outputs from an earlier failed/unfiltered preparation. Old directories are left untouched. If you already configured a run with a different helper version, use a fresh run directory; do not edit the recorded hashes to bypass cache checks.


In [ ]:
DATASET = "webqsp"
PREPARED = ROOT / "output" / f"{DATASET}-prepared-graph-only"
RUN = ROOT / "output" / f"{DATASET}-retrieval-fp32-graph-only"
if not (PREPARED / "prepared.json").exists():
    revision = HfApi().dataset_info('rmanluo/RoG-webqsp').sha
    command = [PY, "-u", str(SCRIPT), "prepare", "--dataset", DATASET,
               "--output", str(PREPARED)]
    command += ['--revision', revision]
    subprocess.run(command, cwd=ROOT, check=True)
metadata, records = pipeline.read_prepared(PREPARED)
print("Source:", metadata["source"])
print("Source / retained / excluded samples:", metadata["source_count"], metadata["count"], metadata["excluded_count"])
print("Splits:", {name: len(ids) for name, ids in metadata["splits"].items()})
print("Original split sizes:", {name: len(ids) for name, ids in metadata["source_splits"].items()})
print("Exclusion audit:", PREPARED / "excluded.json")
print("First retained source mapping:", {key: records[0][key] for key in ("id", "source_index", "source_id", "source_split", "source_split_index")})
print("Question:", records[0]["question"])
print("Label:", records[0]["label"])
display(pd.read_csv(PREPARED / "nodes/0.csv", keep_default_na=False).head())
display(pd.read_csv(PREPARED / "edges/0.csv", keep_default_na=False).head())


## 4. Visualize input size and a bounded graph preview


In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

stats = pd.DataFrame(records)[["id", "nodes", "edges"]]
split_by_id = {i: name for name, ids in metadata["splits"].items() for i in ids}
stats["split"] = stats["id"].map(split_by_id)
display(stats.groupby("split")[["nodes", "edges"]].agg(["mean", "max"]))
display(stats[["nodes", "edges"]].describe(percentiles=[0.5, 0.9, 0.99]))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
stats.groupby("split")[["nodes", "edges"]].mean().plot.bar(ax=axes[0], rot=0)
axes[0].set_title("Mean full graph size per query")
axes[1].hist([stats["nodes"], stats["edges"]], bins=40, label=["nodes", "edges"])
axes[1].set_title("Full graph size distribution")
axes[1].legend()
plt.tight_layout()
plt.show()

PREVIEW_INDEX = 0
nodes, edges = pipeline.load_tables(PREPARED, records[PREVIEW_INDEX], DATASET)
shown = set(nodes.node_id.head(30))
preview = nx.DiGraph()
preview.add_nodes_from(sorted(shown))
preview.add_edges_from((row.src, row.dst) for row in edges.itertuples()
                       if row.src in shown and row.dst in shown)
plt.figure(figsize=(12, 6))
labels = {row.node_id: row.node_attr[:28] for row in nodes.itertuples() if row.node_id in shown}
nx.draw_networkx(preview, pos=nx.spring_layout(preview, seed=42),
                 labels=labels, node_size=600, font_size=7, arrows=True)
plt.title(f"Sample {PREVIEW_INDEX}: first {len(shown)} nodes only (display limit, not preprocessing)")
plt.axis("off")
plt.show()


## 5. Configure retrieval, then smoke-test one query on each T4

The encoder is the repository's public all-roberta-large-v1, with attention-mask
mean pooling and L2 normalization. Tokenize **per batch**, deduplicate entity/relation
strings per query, and return CPU FP32 features. Use batch size 16 initially.
`FP16=False` prioritizes numerical fidelity; optional FP16 affects inference only
(pooling and stored features remain FP32). Use a new run directory when settings change.
Source/model revisions and all retrieval settings are recorded; resume refuses mismatches.

A sample is committed atomically only after retrieval and validation. Shards process
disjoint IDs `i % 2`; filesystem locks reject duplicate shard launches. Failed samples
are not silently replaced or skipped. The smoke check is not a full preprocessing run.


In [ ]:
BATCH_SIZE = 16
FP16 = False
if (RUN / "config.json").exists():
    MODEL_REVISION = json.loads((RUN / "config.json").read_text())["model_revision"]
else:
    MODEL_REVISION = HfApi().model_info(pipeline.MODEL).sha
config = pipeline.configure_run(
    PREPARED, RUN, num_shards=2, batch_size=BATCH_SIZE, fp16=FP16,
    k=2, topk=20, topk_entity=10, max_length=512, model_revision=MODEL_REVISION,
)
if len(records) < 2:
    raise ValueError("This two-GPU smoke check requires at least two samples.")
print(json.dumps(config, indent=2))

def run_shards(stop=None, *, script=SCRIPT):
    processes = []
    logs = []
    try:
        for shard in range(config["num_shards"]):
            prefix = "shard" if script == SCRIPT else "benchmark-shard"
            log_path = RUN / f"{prefix}-{shard}.log"
            logs.append(log_path)
            command = [PY, "-u", str(script), "worker",
                       "--output", str(RUN), "--shard", str(shard)]
            if stop is not None:
                command += ["--stop", str(stop)]
            env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(shard),
                   "OMP_NUM_THREADS": "2", "MKL_NUM_THREADS": "2"}
            with log_path.open("a") as log:
                processes.append(subprocess.Popen(
                    command, cwd=ROOT, env=env, stdout=log, stderr=subprocess.STDOUT
                ))
        print("Worker PIDs:", [process.pid for process in processes], flush=True)
        import time
        while True:
            codes = [process.poll() for process in processes]
            completed = len(list((RUN / "samples").glob("*.pt")))
            print(time.strftime("%H:%M:%S"), f"saved={completed}/{len(records)}",
                  "exit codes:", codes, flush=True)
            for path in logs:
                with path.open("rb") as handle:
                    handle.seek(0, 2)
                    handle.seek(max(0, handle.tell() - 1500))
                    print(path.name, handle.read().decode(errors="replace"))
            if any(code not in (None, 0) for code in codes):
                raise RuntimeError(f"A worker failed. Read the logs under {RUN}.")
            if all(code == 0 for code in codes):
                break
            time.sleep(30)
    finally:
        # Stop only these owned PIDs, including on Interrupt; never kill by name.
        for process in processes:
            if process.poll() is None:
                process.terminate()
        for process in processes:
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
    return [process.returncode for process in processes]

assert run_shards(stop=2) == [0, 0]


## 6. Inspect retrieved structure, compression, and model-input shapes

The saved tensor dictionary stores unique selected node/relation features, integer
expansion maps, local concatenated edge indices, **one** query vector, center scores,
source mappings, question, label, and CSV description. Features shared by overlapping
ego-graphs are duplicated only in RAM when a sample is loaded. This storage change is
lossless relative to this run's computed features.

`CompactDataset(RUN)` returns `id/question/label/desc/graph`, with float32
`x`, `edge_attr`, `question_node`, `question_edge` for GraphLLM.
It is an **explicit adapter**, not a drop-in replacement for the legacy `.pt` files
or an automatic change to `train.py`'s dataset registry. No Llama model is loaded here.


In [ ]:
from torch_geometric.data import Batch

dataset = pipeline.CompactDataset(RUN)
sample = dataset[PREVIEW_INDEX]
graph = sample["graph"]
payload = pipeline.load_payload(RUN / "samples" / f"{PREVIEW_INDEX}.pt")
print(graph)
print("Query shape:", tuple(payload["query"].shape))
print("Selected centers:", payload["centers"].tolist())
print("Counts:", payload["stats"])
print("Unique feature rows:", len(payload["node_features"]), len(payload["edge_features"]))
print("Question:", sample["question"])
print("Hard prompt preview:\n", sample["desc"][:1600])
assert graph.x.shape == graph.question_node.shape
assert graph.edge_attr.shape == graph.question_edge.shape
assert graph.edge_index.shape == (2, len(graph.edge_attr))
for tensor in (graph.x, graph.edge_attr, graph.question_node, graph.question_edge):
    assert tensor.dtype == torch.float32 and torch.isfinite(tensor).all()
batch = Batch.from_data_list([dataset[0]["graph"], dataset[1]["graph"]])
print("Two-sample PyG batch:", batch)
del batch

scores = payload["scores"]
best = torch.topk(scores, min(10, len(scores))).indices
plt.figure(figsize=(9, 3))
plt.bar([str(i) for i in best.tolist()], scores[best].numpy())
plt.xlabel("Ego center (original node ID)")
plt.ylabel("Cosine relevance")
plt.title("Highest-scoring K-hop ego-graphs")
plt.show()

sizes = [path.stat().st_size for path in (RUN / "samples").glob("*.pt")]
expanded_bytes = sum(t.numel() * t.element_size() for t in
                     (graph.x, graph.edge_attr, graph.question_node, graph.question_edge,
                      graph.edge_index))
print("Compact sample MiB:", (RUN / "samples" / f"{PREVIEW_INDEX}.pt").stat().st_size / 2**20)
print("Expanded model-input tensors MiB:", expanded_bytes / 2**20)
print("Smoke-based output estimate GiB:", max(sizes) * len(records) / 2**30)
del graph, payload, sample


## 6a. Actual parameter count and measured two-T4 runtime estimate

This benchmarks **preprocessing only**, not Llama training or inference. Each T4 loads one copy of the same SentenceBERT encoder; the logical model parameter count is not doubled. The cell counts the loaded weights directly and reports **zero optimized parameters**, because preprocessing uses inference only. Weight memory excludes activations, CUDA buffers, and graph tensors.

Both GPUs benchmark concurrently using the existing pipeline. For each modulo shard, queries are sorted by raw node+edge count and split into up to three size buckets; the median-sized query in each bucket is measured. Timers synchronize CUDA and separately report embedding, CPU ego-retrieval, and validation/serialization costs. These benchmark queries are computed again without writing/replacing sample caches. Small timing JSON reports are retained under `RUN/benchmark`; models exit with their subprocesses.

The remaining-time estimate weights each measured bucket by its unfinished queries, adds measured model startup/warmup per shard, and takes the **slower shard's total**. It does not assume a perfect 2x speedup. This is a rough point estimate, not an upper bound: token lengths, graph topology, other workloads, and thermal conditions vary. Downloads, CSV preparation/plots, final full-run validation, disk writes, Google Drive uploads, and the benchmark/monitor overhead itself are excluded. The GPU names show whether the measurements really came from T4s. Rerun this cell to refresh the estimate after more samples complete.


In [ ]:
METRICS_SCRIPT = SCRIPT.with_name("preprocessing_metrics.py")
if not METRICS_SCRIPT.is_file():
    raise FileNotFoundError("Upload/pull preprocessing_metrics.py beside the pipeline helper first.")
assert run_shards(script=METRICS_SCRIPT) == [0, 0]
metrics_spec = importlib.util.spec_from_file_location("preprocessing_metrics", METRICS_SCRIPT)
metrics = importlib.util.module_from_spec(metrics_spec)
metrics_spec.loader.exec_module(metrics)
timing = metrics.estimate_remaining(RUN)
print(f"Encoder parameters (one logical model): {timing['parameters_per_encoder']:,}")
print("Optimized parameters during preprocessing:", timing["optimized_parameters"])
print("Measured GPUs:", timing["gpus"])
print("Weight-only GiB per GPU:", timing["weight_gib_per_gpu"])
if not all("T4" in name for name in timing["gpus"]):
    print("These are measurements for the listed hardware, not a T4 runtime prediction.")
display(pd.DataFrame(timing["observations"])[[
    "shard", "sample_id", "embedding_seconds", "retrieval_seconds",
    "table_seconds", "validation_serialization_seconds", "total_seconds",
]])
print("Unfinished queries:", timing["remaining_samples"])
print("Estimated seconds per shard:", timing["estimated_seconds_per_shard"])
print(f"Rough remaining preprocessing wall time: {timing['estimated_wall_seconds'] / 60:.1f} minutes "
      f"({timing['estimated_wall_seconds'] / 3600:.2f} hours); exclusions/uncertainty above apply.")


## 7. Full run: opt in only after checking disk and runtime

Change `RUN_FULL` to `True` to process all rows. Completed valid samples are reused.
The two-sample estimate is **not an upper bound**: graph sizes are skewed. The cell
also scales smoke sizes by the largest full graph's node+edge count for a rough
headroom estimate, not a guarantee. Workers stop if a write would leave under 2 GiB
free. This does not impose graph-size caps or guarantee that all data fits in Kaggle.
Retrieval still scans all node-centered K-hop neighborhoods, and can be slow.
Interrupting the launch/monitor cell terminates its two owned workers; completed
samples survive. Do not start a second notebook against the same run directory.
After a kernel restart inspect recorded PIDs/logs before relaunching; a worker lock
error means an old worker is still alive. Never use blanket process-name termination.


In [ ]:
import shutil

RUN_FULL = False
smoke_sizes = [(RUN / "samples" / f"{i}.pt").stat().st_size for i in range(2)]
source_sizes = [max(1, row["nodes"] + row["edges"]) for row in records]
largest_scale = max(source_sizes) / min(source_sizes[:2])
estimate = max(smoke_sizes) * largest_scale * len(records)
free = shutil.disk_usage(RUN).free
print(f"Rough size-scaled estimate: {estimate / 2**30:.2f} GiB; free: {free / 2**30:.2f} GiB")
if RUN_FULL:
    if estimate + 2 * 2**30 > free:
        raise OSError("Estimated outputs exceed headroom. Save/inspect this run before proceeding.")
    assert run_shards() == [0, 0]
    subprocess.run([PY, "-u", str(SCRIPT), "validate", "--output", str(RUN)],
                   cwd=ROOT, check=True)
else:
    print("Smoke check only. Set RUN_FULL=True here when ready.")


## 8. Persist compact outputs and provenance

Save the `output/<dataset>-prepared` and `output/<dataset>-retrieval-fp32` directories
as a Kaggle Dataset/versioned notebook output. No full embedded base graphs need to
be retained. Do not export the HF model cache, tokens, or old `dataset/*/graphs` caches.
Existing legacy graph files are **not deleted** by this notebook; inspect their disk
usage separately if reusing a previous Kaggle workspace.

To resume, copy your saved prepared/run directories from read-only `/kaggle/input`
back to the **same `/kaggle/working/BioGRAG-QA/output/...` paths**, restore the same
helper version, then run this notebook from the top. Do not write into `/kaggle/input`.
Config/source hashes prevent accidentally mixing settings. Split files contain contiguous processing IDs for the retained graph-only subset; records and the exclusion audit preserve original source IDs and split membership. A smoke subset is not a trainable full dataset.


In [ ]:
provenance = {
    "source_commit": COMMIT,
    "helper_sha256": pipeline.digest(SCRIPT),
    "dataset_source": metadata["source"],
    "source_samples": metadata["source_count"],
    "retained_samples": metadata["count"],
    "excluded_samples": metadata["excluded_count"],
    "python": sys.version,
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpus": [torch.cuda.get_device_name(i) for i in range(2)],
    "saved_samples": len(list((RUN / "samples").glob("*.pt"))),
    "expected_samples": len(records),
    "config": config,
}
provenance_path = RUN / f"session-{os.getpid()}-provenance.json"
pipeline.atomic_json(provenance, provenance_path)
with (RUN / f"session-{os.getpid()}-packages.txt").open("w") as handle:
    subprocess.run([PY, "-m", "pip", "freeze"], stdout=handle, check=True)
print("Save prepared text data:", PREPARED)
print("Save compact outputs:", RUN)
print("Recorded provenance:", provenance_path)


## 9. Optional automatic Google Drive backup

Complete the one-time **personal Drive OAuth setup** in [the README](../../dataset/webqsp/README.md#google-drive-backup-optional), then enable these three Kaggle Secrets for this notebook: `GDRIVE_CLIENT_ID`, `GDRIVE_CLIENT_SECRET`, `GDRIVE_REFRESH_TOKEN`. Never paste their values into cells, outputs, Git, or chat. Kaggle does not support Colab's `drive.mount()`.

Set `ENABLE_DRIVE_BACKUP=True` below **before Run All**. Once preprocessing finishes, this cell validates the entire dataset and uploads it to the private, app-created **GRAG preprocessing backups** folder in your Drive. Smoke-only/incomplete runs are rejected. This is a versioned backup, not a destructive mirror or a guarantee against Kaggle terminating a session before the final cell runs.

Uploads use resumable requests and verified size/MD5 checks. Only one approximately 256 MiB source-data archive part exists locally at a time (a single larger file can exceed that target); it is removed after upload. The manifest is uploaded last, marking a completed snapshot. Retry the cell after an upload error; matching completed parts are reused. Outputs are never deleted locally. Logs, package listings, credentials, HF weights, and legacy full-graph caches are excluded.

For downstream LLM training, download **every part plus manifest.json**. Verify the SHA256 for each part against the manifest, then extract the parts at the repository root: they contain `output/<prepared-folder>/...` and `output/<run-folder>/...`. Restore the same helper version and working paths listed in the manifest, then use `pipeline.CompactDataset(RUN)` to recover graphs, questions, labels, descriptions, and original splits. The existing training registry still needs that explicit adapter.


In [ ]:
ENABLE_DRIVE_BACKUP = False  # Set True after configuring Kaggle Secrets.
if ENABLE_DRIVE_BACKUP:
    pipeline.validate_run(RUN)
    BACKUP_SCRIPT = SCRIPT.with_name("drive_backup.py")
    if not BACKUP_SCRIPT.is_file():
        raise FileNotFoundError("Upload/pull the shared drive_backup.py helper first.")
    drive_packages = {"google.auth": "google-auth", "googleapiclient": "google-api-python-client", "requests": "requests"}
    drive_missing = []
    for module, package in drive_packages.items():
        try:
            present = importlib.util.find_spec(module) is not None
        except ModuleNotFoundError:
            present = False
        if not present:
            drive_missing.append(package)
    if drive_missing:
        subprocess.run([PY, "-m", "pip", "install", "-q", *drive_missing], check=True)
        importlib.invalidate_caches()
    from kaggle_secrets import UserSecretsClient
    from google.oauth2.credentials import Credentials
    from google.auth.transport.requests import Request
    from googleapiclient.discovery import build

    drive_spec = importlib.util.spec_from_file_location("drive_backup", BACKUP_SCRIPT)
    drive_backup = importlib.util.module_from_spec(drive_spec)
    drive_spec.loader.exec_module(drive_backup)
    secrets = UserSecretsClient()
    credentials = Credentials(
        token=None,
        refresh_token=secrets.get_secret("GDRIVE_REFRESH_TOKEN"),
        token_uri="https://oauth2.googleapis.com/token",
        client_id=secrets.get_secret("GDRIVE_CLIENT_ID"),
        client_secret=secrets.get_secret("GDRIVE_CLIENT_SECRET"),
        scopes=[drive_backup.DRIVE_SCOPE],
    )
    credentials.refresh(Request())
    service = build("drive", "v3", credentials=credentials, cache_discovery=False)
    try:
        DRIVE_BACKUP_URL = drive_backup.backup_to_drive(PREPARED, RUN, service)
    finally:
        service.close()
        del service, credentials, secrets
else:
    print("Drive backup disabled. Local outputs remain available for Kaggle Save Version.")
